# Clasificador de botellas con inventario
Cuaderno propio basado en la CNN del ejemplo de perros y gatos. Se entrena desde cero con nuestras fotos; no reutiliza pesos de animales.

**Flujo:** fotos → entrenamiento → evaluación → exportación → página de inventario.

Diferencias: imágenes a color de 100×100, salida softmax para varias clases, clase `otro` y exportación automática de `metadata.json`. El inventario sigue admitiendo cualquier producto manualmente; solo se reconocen los productos entrenados.

## 1. Preparar fotos
Crea `dataset.zip` con estas carpetas directamente en la raíz:
```
coca_cola_500/  (fotos de Coca-Cola de esa presentación)
vital_600/      (fotos de Agua Vital de esa presentación)
otro/           (mesa, manos, vasos y otras botellas)
```
Puedes agregar más carpetas, una por producto/presentación. Usa los nombres de clase del catálogo de la app. Recomendación: al menos 100 fotos distintas por clase, variando luz, fondo, ángulo y distancia. No uses cuadros casi idénticos de un mismo video en entrenamiento y prueba. Reserva sesiones de captura diferentes para una prueba adicional con el celular.

Las fotografías de una botella nueva deben agregarse al dataset y hay que volver a entrenar para reconocerla. `otro` es un rechazo aprendido; no garantiza detectar todos los objetos desconocidos.

## 2. Abrir y ejecutar
Sube este archivo a Google Colab. Selecciona GPU si está disponible. Ejecuta las celdas en orden. La instalación cambia dependencias solo en Colab, no en tu proyecto local. Si Colab solicita reiniciar la sesión tras instalar, reiníciala y continúa con la celda de importaciones.


In [ ]:
%pip install -q tensorflowjs==4.22.0 tf-keras pillow matplotlib scikit-learn


In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
from pathlib import Path
import json, zipfile, hashlib, shutil
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import tensorflow as tf
import tf_keras as keras
import tensorflowjs as tfjs

SEMILLA = 42
TAMANO = 100
EPOCAS = 40
keras.utils.set_random_seed(SEMILLA)
print('TensorFlow:', tf.__version__, 'GPU:', tf.config.list_physical_devices('GPU'))


## 3. Subir el dataset
Esta celda pide el ZIP desde tu computadora. Usa un nombre de carpeta por clase, sin una carpeta contenedora adicional.

In [ ]:
from google.colab import files
subidos = files.upload()
assert len(subidos) == 1, 'Sube solo dataset.zip.'
nombre_zip = next(iter(subidos))
RAIZ = Path('/content/dataset_botellas')
assert not RAIZ.exists(), 'La carpeta ya existe. Usa una sesión nueva para evitar mezclar datasets.'
RAIZ.mkdir()
with zipfile.ZipFile(nombre_zip) as z:
    for item in z.infolist():
        destino = (RAIZ / item.filename).resolve()
        assert destino.is_relative_to(RAIZ.resolve()), 'Ruta inválida en ZIP.'
    z.extractall(RAIZ)
del subidos
CLASES = sorted(p.name for p in RAIZ.iterdir() if p.is_dir() and not p.name.startswith(('.', '__')))
assert len(CLASES) >= 3 and 'otro' in CLASES, 'Incluye productos y la carpeta otro.'
print('Orden fijo de clases:', dict(enumerate(CLASES)))


## 4. Leer y separar imágenes
Se eliminan duplicados exactos y se reserva 15 % para validación y 15 % para prueba por clase. La prueba no interviene en el entrenamiento. Fotografías visualmente similares pueden seguir causando fuga: separa las sesiones de captura antes de reunir el ZIP.

In [ ]:
imagenes, etiquetas, vistos = [], [], {}
for indice, clase in enumerate(CLASES):
    cantidad = 0
    for archivo in sorted((RAIZ / clase).rglob('*')):
        if archivo.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.webp'}: continue
        with Image.open(archivo) as original:
            rgb = ImageOps.exif_transpose(original).convert('RGB')
            huella = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
            if huella in vistos:
                assert vistos[huella] == clase, f'La misma foto está etiquetada en dos clases: {archivo}'
                continue
            vistos[huella] = clase
            rgb = rgb.resize((TAMANO, TAMANO), Image.Resampling.BILINEAR)
            imagenes.append(np.asarray(rgb, dtype=np.float32) / 255.0)
            etiquetas.append(indice); cantidad += 1
    print(clase, cantidad, 'fotos únicas')
    assert cantidad >= 30, f'Faltan fotos en {clase}; mínimo técnico 30, recomendado 100 o más.'
X, y = np.stack(imagenes), np.array(etiquetas)
indices = np.arange(len(y))
train_idx, resto = train_test_split(indices, test_size=.30, stratify=y, random_state=SEMILLA)
val_idx, test_idx = train_test_split(resto, test_size=.50, stratify=y[resto], random_state=SEMILLA)
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]
print('Entrenamiento:', len(y_train), 'Validación:', len(y_val), 'Prueba:', len(y_test))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, i in zip(axes.flat, train_idx[:8]):
    ax.imshow(X[i]); ax.set_title(CLASES[y[i]]); ax.axis('off')
plt.tight_layout()


## 5. Crear la CNN
Como el ejemplo: tres convoluciones, agrupamiento, capa densa y salida. Usamos color para conservar información de etiquetas y tapas, y una salida por clase en lugar de una salida binaria. El aumento de datos se aplica solo al entrenamiento, fuera del modelo exportado.

In [ ]:
aumento = keras.preprocessing.image.ImageDataGenerator(
    rotation_range=15, width_shift_range=.1, height_shift_range=.1,
    zoom_range=.15)
modelo = keras.Sequential([
    keras.layers.InputLayer(input_shape=(TAMANO, TAMANO, 3)),
    keras.layers.Conv2D(32, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(128, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(100, activation='relu'),
    keras.layers.Dropout(.4),
    keras.layers.Dense(len(CLASES), activation='softmax'),
], name='clasificador_botellas')
modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=.001),
               loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelo.summary()


## 6. Entrenar
Las épocas son un máximo. Se detiene cuando deja de mejorar la validación y se recuperan los mejores pesos. No se garantiza un porcentaje de aciertos antes de medirlo.

In [ ]:
historia = modelo.fit(
    aumento.flow(X_train, y_train, batch_size=16, seed=SEMILLA),
    validation_data=(X_val, y_val), epochs=EPOCAS,
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
               keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=3, factor=.5)])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metrica in zip(axes, ['accuracy', 'loss']):
    ax.plot(historia.history[metrica], label='Entrenamiento')
    ax.plot(historia.history['val_' + metrica], label='Validación')
    ax.set_title(metrica); ax.set_xlabel('Época'); ax.legend()
plt.tight_layout()


## 7. Medir y fijar el umbral
Se usa un umbral inicial de 70 %. Si lo ajustas, utiliza solo validación; conserva prueba para la evaluación final. Una confianza alta no garantiza que el objeto pertenezca a una clase conocida.

In [ ]:
UMBRAL = .70
probabilidades = modelo.predict(X_test, verbose=0)
predicciones = probabilidades.argmax(axis=1)
print(classification_report(y_test, predicciones, labels=range(len(CLASES)), target_names=CLASES, zero_division=0))
matriz = confusion_matrix(y_test, predicciones, labels=range(len(CLASES)))
ConfusionMatrixDisplay(matriz, display_labels=CLASES).plot(xticks_rotation=45)
plt.show()
aceptados = (probabilidades.max(axis=1) >= UMBRAL) & (predicciones != CLASES.index('otro'))
print('Reconocimientos aceptados:', int(aceptados.sum()), '/', len(y_test))
if aceptados.any(): print('Aciertos entre aceptados:', float(np.mean(predicciones[aceptados] == y_test[aceptados])))
metricas = classification_report(y_test, predicciones, labels=range(len(CLASES)), target_names=CLASES, output_dict=True, zero_division=0)


## 8. Exportar para nuestra aplicación
Se generan `model.json`, los fragmentos `.bin` y `metadata.json` con el orden real de clases y el preprocesamiento. El número de archivos binarios depende del tamaño del modelo. La carpeta de entrenamiento también conserva métricas y un modelo H5. No se sobrescriben los archivos locales de D: desde Colab.

In [ ]:
SALIDA = Path('/content/exportacion_botellas')
assert not SALIDA.exists(), 'Ya exportaste. Usa otra carpeta o una sesión nueva para conservar esa exportación.'
CARPETA_MODELO = SALIDA / 'modelo'
CARPETA_MODELO.mkdir(parents=True)
tfjs.converters.save_keras_model(modelo, str(CARPETA_MODELO))
metadata = {
    'nombre': 'Clasificador de botellas',
    'descripcion': 'CNN propia entrenada con fotos del grupo',
    'modelo': 'model.json',
    'entrada': {'alto': TAMANO, 'ancho': TAMANO, 'canales': 3},
    'preprocesamiento': {'escala': '0-1', 'recorte': 'estirar'},
    'salida': 'softmax', 'confianzaMinima': UMBRAL,
    'clases': [{'id': c, 'nombre': c.replace('_', ' ')} for c in CLASES],
    'enVivo': {'fps': 4}
}
(CARPETA_MODELO / 'metadata.json').write_text(json.dumps(metadata, indent=2, ensure_ascii=False))
(SALIDA / 'metricas.json').write_text(json.dumps(metricas, indent=2))
modelo.save(str(SALIDA / 'botellas.h5'))
# Verificar que estén todos los pesos y la cantidad de clases exportadas.
exportado = json.loads((CARPETA_MODELO / 'model.json').read_text())
for grupo in exportado['weightsManifest']:
    for ruta in grupo['paths']: assert (CARPETA_MODELO / ruta).is_file(), ruta
assert len(metadata['clases']) == modelo.output_shape[-1]
shutil.make_archive('/content/botellas_entrenadas', 'zip', SALIDA)
files.download('/content/botellas_entrenadas.zip')


## 9. Integrar en D: y probar
1. Guarda un respaldo del modelo anterior fuera de la carpeta publicada.
2. Reemplaza el contenido de `Botellas inventario/modelo/` por la carpeta `modelo/` del ZIP exportado. Mantén `metricas.json` y `botellas.h5` en `entrenamiento/`.
3. Abre `iniciar_servidor.bat` y recarga con Ctrl+F5. Debe aparecer «Modelo listo».
4. Los identificadores de clase deben coincidir con `data/productos.json`. Para los productos añadidos desde el formulario, su identificador se puede consultar en la exportación JSON; usa ese identificador como nombre de su carpeta de fotos para que la clase coincida.
5. Prueba fotos nuevas de cada producto y de objetos desconocidos. En cámara, coloca la botella centrada: el visor recorta al centro.
6. Comprueba nombre, stock, venta, entrada y reposición. El reconocimiento no modifica stock por sí solo.
7. Para la cámara del celular, publica con HTTPS y ensaya con el dispositivo real.

No presentes como real un reconocimiento simulado. Si confunde productos, amplía y mejora el dataset y vuelve a entrenar; no cambies las etiquetas para ocultar errores.

### Fuentes
- Cuaderno local de referencia: `Ejemplo ING/entrenamiento/Clasificador_de_perros_y_gatos.ipynb`.
- Exportación Keras → TensorFlow.js: https://www.tensorflow.org/js/guide/conversion
